In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.removeAll()

In [0]:
## PARAMETRIZAR CATALOGO A PROD
dbutils.widgets.text("PRM_catalogo","catalogo_desa_intEcommerce")

PRM_catalogo = dbutils.widgets.get("PRM_catalogo")

In [0]:
def read_tablas_Silver():

    df_tablaproducto = spark.table(f"{PRM_catalogo}.silver.Tabla_Producto") \
        .select(
            col("Cod_producto").alias("Cod_producto"),
            col("Nombre_producto").alias("Nombre_producto"),
            col("Categoria").alias("Categoria")
        )
        
    df_cliente = spark.table(f"{PRM_catalogo}.silver.Tabla_Cliente") \
        .select(
            col("ID_Cliente").alias("ID_Cliente"), 
            concat(col("Nombre"), lit(" "), col("Apellido")).alias("Nombre_completo"),
            col("Numero_Celular").alias("Numero_Celular")          
        ) 

    df_TablaEcommerce = spark.table(f"{PRM_catalogo}.silver.Tabla_IntEcommerce") \
        .select(
            col("ID_Cliente").alias("ID_Cliente"),
            trim(col("ID_interaccion")).alias("ID_interaccion"),
            date_format(to_timestamp(col("Fecha_Interaccion"), "M/d/yyyy H:mm"), "MM-yyyy").alias("Periodo_Mes"),
            date_format(to_timestamp(col("Fecha_Interaccion"), "M/d/yyyy H:mm"), "dd/MM/yyyy").alias("Fecha_Interaccion"),
            date_format(to_timestamp(col("Fecha_Interaccion"), "M/d/yyyy H:mm"), "HH:mm").alias("Hora_Interaccion"),
            trim(col("Cod_producto")).alias("Cod_producto"),
            trim(col("Evento")).alias("Evento"),
            trim(col("Locacion")).alias("Locacion"),
            trim(col("Cod_Tipo_Interaccion")).alias("Cod_Tipo_Interaccion"),
            col("Cantidad_Producto").alias("Cantidad_Producto"),
            col("Puntuacion_Producto").alias("Puntuacion_Producto")         
        ) \
        .where(
                (col("ID_interaccion").isNotNull()) 
              )
              
    df_TipInteraccion = spark.table(f"{PRM_catalogo}.silver.Tabla_Destipinteraccion") \
        .select(
            trim(col("Cod_Tipo_Interaccion")).alias("Cod_Tipo_Interaccion"),  
            trim(col("Nombre_codigo_sistema")).alias("Nombre_codigo_sistema"),
            trim(col("Descripcion_interaccion")).alias("Descripcion_interaccion") 
        )

    return df_tablaproducto,df_cliente, df_TablaEcommerce, df_TipInteraccion

In [0]:
def tablajoin(df_tablaproducto, df_cliente, df_TablaEcommerce, df_TipInteraccion):

    df_join = df_TablaEcommerce.alias("A")\
        .join(df_cliente.alias("C"), col("A.ID_Cliente") == col("C.ID_Cliente"), "left")\
        .join(df_tablaproducto.alias("P"), col("A.Cod_producto") == col("P.Cod_producto"), "left")\
        .join(df_TipInteraccion.alias("I"), col("A.Cod_Tipo_Interaccion") == col("I.Cod_Tipo_Interaccion"), "left")\
            .select(
                col("A.ID_interaccion").alias("ID_interaccion"),
                col("A.Periodo_Mes").alias("Periodo_Mes"),
                to_date(col("A.Fecha_Interaccion"), "dd/MM/yyyy").alias("Fecha_Interaccion"),
                col("A.Hora_Interaccion").alias("Hora_Interaccion"),
                col("A.ID_Cliente").alias("ID_Cliente"),	
                col("C.Nombre_completo").alias("Nombre_completo"),
                col("C.Numero_Celular").alias("Numero_Celular"),
                col("I.Nombre_codigo_sistema").alias("Nombre_CodInteraccion"),
                col("I.Descripcion_interaccion").alias("Descripcion_interaccion"),                 
                col("A.Evento").alias("Evento"), 
                col("A.Locacion").alias("Locacion"),
                col("P.Categoria").alias("Categoria"),
                col("P.Nombre_producto").alias("Nombre_producto"),
                col("A.Cantidad_Producto").alias("Cantidad_Producto"),
                col("A.Puntuacion_Producto").alias("Puntuacion_Producto")
            ).where(col("A.ID_interaccion").isNotNull())

    return df_join

In [0]:
def main():

    df_tablaproducto, df_cliente, df_TablaEcommerce, df_TipInteraccion = read_tablas_Silver()
    
    df_join = tablajoin(df_tablaproducto, df_cliente, df_TablaEcommerce, df_TipInteraccion)
    
    df_join.write.mode("overwrite").saveAsTable(f"{PRM_catalogo}.golden.Interaccion_Analisis")

main()